# Lab Instructions

In the lab, you're presented a task such as building a dataset, training a model, or writing a training loop, and we'll provide the code structured in such a way that you can fill in the blanks in the code using the knowledge you acquired in the chapters that precede the lab. You should be able to find appropriate snippets of code in the course content that work well in the lab with minor or no adjustments.

The blanks in the code are indicated by ellipsis (`...`) and comments (`# write your code here`).

In some cases, we'll provide you partial code to ensure the right variables are populated and any code that follows it runs accordingly.

```python
# write your code here
x = ...
```

The solution should be a single statement that replaces the ellipsis, such as:

```python
# write your code here
x = [0, 1, 2]
```

In some other cases, when there is no new variable being created, the blanks are shown like in the example below:

```python
# write your code here
...
```

Although we're showing you only a single ellipsis (`...`), you may have to write more than one line of code to complete the step, such as:

```python
# write your code here
for i, xi in enumerate(x):
    x[i] = xi * 2
```

## Installation Notes

This notebook requires a **GPU runtime** in Google Colab.

The solution uses current Transformers, bitsandbytes, PEFT, and TRL APIs.

In [ ]:
# Install the packages needed for quantized loading, LoRA, and supervised fine-tuning.
!pip install -q -U accelerate bitsandbytes peft trl evaluate

In [ ]:
import torch

# Stop early if the runtime cannot train and run the model on a CUDA GPU.
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before running this lab.")

# Confirm which GPU the notebook will use.
print(torch.cuda.get_device_name(0))

## 18.25 Lab 8: Fine-Tuning an LLM for Structured Information Extraction

This is the capstone lab for Part IV. You will adapt a small base causal language model to convert natural-language calendar requests into a constrained JSON record containing:
- an **action**: `set`, `query`, or `remove`;
- zero or more extracted **slots**.

Unlike a simple intent classifier, the model must jointly:
1. identify the operation;
2. extract a variable number of values;
3. obey a fixed output schema;
4. avoid inventing missing information.

### 18.25.1 Dataset

The [MASSIVE](https://huggingface.co/datasets/AmazonScience/massive) dataset provides official train, validation, and test splits. The dataset was originally distributed through a Hugging Face dataset loading script. Current versions of the datasets library no longer execute dataset scripts, so we'll load the English Parquet files directly from the Hub instead. The resulting DatasetDict behaves exactly like any other Hugging Face dataset.

In [ ]:
from datasets import load_dataset

# this is how you'd load it, if it were a regular dataset
# massive = load_dataset("AmazonScience/massive", "en-US")

# Point each dataset split to its English-language Parquet shard on Hugging Face.
data_files = {
    "train": "hf://datasets/AmazonScience/massive@refs/convert/parquet/en-US/train/0000.parquet",
    "validation": "hf://datasets/AmazonScience/massive@refs/convert/parquet/en-US/validation/0000.parquet",
    "test": "hf://datasets/AmazonScience/massive@refs/convert/parquet/en-US/test/0000.parquet",
}
# Load the split files into a DatasetDict.
massive = load_dataset("parquet", data_files=data_files)
massive

Let's inspect a training example to identify the fields used later.

In [ ]:
print("\nExample:")
massive["train"][0]

The features that are more interesting to us in this lab are:
- `utt` is the natural-language utterance.
- `annot_utt` contains inline slot annotations.
- `scenario` identifies a broad domain such as calendar or weather.
- `intent` identifies the requested operation.

Let's also inspect the dataset schema and categorical feature definitions. Notice that some columns, such as `scenario` and `intent`, are stored as `ClassLabel`, so they carry a list of possible unique values (names).

In [ ]:
massive["train"].features

We can use these special features to map IDs and names back and forth using `str2int()` and `int2str()` methods.

In [ ]:
# Keep the scenario feature to translate between numeric IDs and labels.
scenario_feature = massive["train"].features['scenario']
# Check that the feature maps scenario names to IDs and back.
(scenario_feature.str2int('calendar'), scenario_feature.int2str(2))

In [ ]:
# Keep the intent feature for decoding the action labels.
intent_feature = massive["train"].features['intent']
intent_feature

#### 18.25.1.1 Selecting the Calendar Scenario

In order to make the dataset size more manageable, we're focusing exclusively on the `calendar` scenario. So, let's filter the dataset accordingly (keep in mind that the values are stored as IDs), hence filtering should use IDs instead of names.

In [ ]:
# Select calendar-related requests in every split using the encoded scenario ID.
calendar_id = scenario_feature.str2int("calendar")
calendar = massive.filter(lambda example: example["scenario"] == calendar_id)
calendar

Although there are several different intents are listed in the corresponding feature, we must check which ones are actually still present in the dataset after filtering:

In [ ]:
# Find the distinct intent IDs present in calendar training examples.
calendar_intents = set(list(calendar['train']['intent']))
calendar_intents

We can use the `int2str()` method to retrieve their corresponding names.

In [ ]:
# Translate the calendar intent IDs into readable operation names.
intent_feature.int2str(calendar_intents)

#### 18.25.1.2 Intent Distribution

How balanced is the resulting dataset? Over each partition, let's find out how many examples of each one of the intents we have:

In [ ]:
import pandas as pd
from collections import Counter

# Count each calendar intent separately for training, validation, and testing.
counts = {}
for split in ["train", "validation", "test"]:
  counts[split] = Counter(list(map(intent_feature.int2str,
                                   calendar[split]['intent'])))

# Arrange the split-level counts into a comparison table.
intent_counts = pd.DataFrame.from_dict(counts)
intent_counts

The exact counts are worth inspecting rather than assuming that the three operations are perfectly balanced. Any severe imbalance can affect both training and per-action evaluation.

### 18.25.2 Parsing the Slot Annotations

The MASSIVE dataset marks spans using syntax such as:

```text
[time : five in the morning]
[date : this week]
```

We preserve values verbatim instead of inventing normalized dates or times. The helper function below uses regular expressions (regex) to find square brackets and retrieve its values, so a JSON object can be returned.

In [ ]:
import re
from collections import defaultdict

# Match MASSIVE slot annotations of the form [slot_name : text].
slot_pattern = re.compile(r"\[([^:\]]+)\s*:\s*([^\]]+)\]")

# Extract annotated slot values, grouping repeated slot names into lists.
def extract_slots(annotated_utterance):
    # Accumulate values by label so multiple mentions of one slot are retained.
    slots = defaultdict(list)
    # Clean each captured label and value before accepting it.
    for label, value in (slot_pattern.findall(annotated_utterance)):
        label = label.strip()
        value = value.strip()
        if label and value:
            slots[label].append(value)

    # Return a plain dictionary suitable for JSON serialization.
    return dict(slots)

Let's double-check the helper function by using a synthetic example first:

In [ ]:
# Self-check using a synthetic annotated utterance.
# Check that repeated person slots are preserved and annotations are parsed correctly.
synthetic = "schedule [event_name : lunch] with [person : Alice] and [person : Bob] [date : tomorrow]"
assert extract_slots(synthetic) == {"event_name": ["lunch"], "person": ["Alice", "Bob"], "date": ["tomorrow"]}
print("Slot parser check passed.")

Then, let's apply it to a few examples in the training set:

In [ ]:
# Compare raw annotations with extracted slots on a few real requests.
for i, example in enumerate(calendar["train"].select(range(5))):
    print(f"\nExample #{i+1}")
    print(example["annot_utt"])
    print(extract_slots(example["annot_utt"]))

#### 18.25.2.1 Discovering the Calendar Schema from Training Data

Which slots are valid? The allowed slot vocabulary should be derived **only from the training split**. The helper function below extracts the slot keys from the annotations for each entry in the dataset.

In [ ]:
# Discover the unique slot labels used in a dataset, returned in sorted order.
def get_slots(dataset):
  # Parse each annotation and collect its slot names into a set.
  return sorted({slot for example in dataset
                 for slot in extract_slots(example["annot_utt"]).keys()})

If we call the function using our training data, we'll define the annotated slots that are allowed in our model:

In [ ]:
# Define the permitted output vocabulary using only training data.
allowed_slots = get_slots(calendar['train'])
print("Allowed slots:", allowed_slots)
print("Number of slot types:", len(allowed_slots))

Are there any slots that can be found either in validation or test splits but that are not part of the training set?

In [ ]:
# Find slot labels in each split to detect labels absent from training.
train_slots = set(allowed_slots)
validation_slots = set(get_slots(calendar["validation"]))
test_slots = set(get_slots(calendar["test"]))
# Report any validation or test labels that the fixed schema cannot express.
print("Unseen validation slots:", validation_slots - train_slots)
print("Unseen test slots:", test_slots - train_slots)

If production data introduced a new slot type, the current fixed schema could not represent it correctly. That would require an explicit schema/version decision and, normally, new training/evaluation examples rather than silently forcing the value into an existing field.

#### 18.25.2.2 Building the Target JSON

We want to train a model to produce the slots as an output, so let's write a function that builds a JSON object as target, as a deterministic serialization reduces irrelevant variation in the training targets. Key ordering and whitespace should not become part of the task.

In [ ]:
import json

# Map the dataset intent names to the three actions required by our output schema.
ACTION_MAP = {
    "calendar_set": "set",
    "calendar_query": "query",
    "calendar_remove": "remove",
}

# Turn a labeled utterance into the expected action-and-slots dictionary.
def build_target(example):
    # Decode the action label and extract annotated slots without normalizing their text.
    intent_name = intent_feature.int2str(example["intent"])
    return {"action": ACTION_MAP[intent_name],
            "slots": extract_slots(example["annot_utt"])}

# Serialize the target consistently for prompt completions and comparisons.
def target_json(example):
    # Sort keys and remove optional whitespace to keep target formatting deterministic.
    return json.dumps(build_target(example), sort_keys=True, separators=(",", ":"))

Let's apply it to a few examples from our training data:

In [ ]:
# Inspect how a few original requests become structured JSON targets.
for i, example in enumerate(calendar["train"].select(range(5))):
    print(f"\nExample #{i+1}")
    print(example["utt"])
    print(target_json(example))

### 18.25.3 Defining the Conversation Structure

We start by writing a system prompt that specifies the task, the format, the allowed slots, and the generation rules.

In [ ]:
# Specify the JSON contract, permitted slot names, and the rule against inferred values.
SYSTEM_PROMPT = f"""
Convert calendar requests into JSON.

Return JSON only.

The output must have exactly this structure:
{{
  "action": "set|query|remove",
  "slots": {{}}
}}

Allowed slot names:
{", ".join(allowed_slots)}

Every slot value must be a list of strings.
Copy values from the request verbatim.
Omit slots that are not present.
Do not infer missing information.
""".strip()

print(SYSTEM_PROMPT)

#### 18.25.3.1 Building Conversational Prompt-Completion Data

Thenm we build a promot-completion set of messages. The prompt contains the system instruction and user request. The completion contains only the target assistant JSON.

In [ ]:
# Format an example as system/user prompt messages plus a JSON assistant completion.
def to_prompt_completion(example):
    # Build the chat-style prompt separately from the expected assistant response.
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["utt"]},
        ],
        "completion": [{"role": "assistant", "content": target_json(example)}],
    }

We can then apply these to our dataset, and remove the existing columns, as the prompt-completion messages are the only required input for the tokenizer.

In [ ]:
# Convert the training and validation splits into SFT examples.
train_data = calendar["train"].map(to_prompt_completion,
                                   remove_columns=calendar["train"].column_names)
validation_data = calendar["validation"].map(to_prompt_completion,
                                             remove_columns=calendar["validation"].column_names)
# Inspect one formatted example before tokenizing it during training.
train_data[1]

### 18.25.4 Loading the Model and Chat Template

We'll use the SmolLM2-360M-Instruct model for our fine-tuning task. We start by loading its tokenizer.

In [ ]:
from transformers import AutoTokenizer
# Choose the instruction-tuned base checkpoint shared by training and inference.
base_checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"

# write your code here
tokenizer = ...

# Provide a padding token if the tokenizer does not define one.
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.unk_token

# Use right padding when preparing training examples.
tokenizer.padding_side = "right"

#### 18.25.4.1 Four-Bit Loading

Then, we'll load the model itself, but we'll load its 4-bit quantized version using BitsAndBytes.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Use bfloat16 where supported, otherwise float16, for quantized computations.
compute_dtype = (torch.bfloat16
                 if torch.cuda.is_bf16_supported(including_emulation=False)
                 else torch.float16)

# Load four-bit NF4 weights with double quantization to reduce GPU memory use.
# write your code here
bnb_config = ...

# Load the base model using the selected quantization and compute dtype.
# write your code here
model = ...

# Check the loaded model location and approximate memory footprint.
print("Model device:", model.device)
print("Approximate model footprint (MB):", model.get_memory_footprint() / 1024**2)

#### 18.25.4.2 Baseline Generation

We evaluate the base model **before** training. The decoding settings remain deterministic (`do_sample=False`) so the before/after comparison is not confounded by sampling. The helper function, `generate_extraction()`, applies the chat template to the messages, set the model to evaluation/inference mode, and then calls the `generate()` method to produce the output. It keeps and decodes the response tokens only.

In [ ]:
# Wrap a user request with the same extraction instructions used for training.
def extraction_messages(utterance):
    # Keep the task instructions and request in separate chat roles.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": utterance},
    ]
    return messages

# Generate a deterministic response from the model for the supplied chat messages.
def generate_extraction(model, tokenizer, messages, max_new_tokens=256):
    # Apply the tokenizer's chat format and move input tensors to the model device.
    # Do not forget to add the generation prompt and padding
    # write your code here
    inputs = ...

    # Remember the prompt length so it can be excluded from the returned response.
    prompt_length = inputs["input_ids"].shape[-1]
    model.eval()

    # Disable gradient tracking and use greedy decoding for reproducible predictions.
    # Do not forget to set the pad_token_id
    with torch.inference_mode():
        # write your code here
        outputs = ...

    # Remove prompt tokens and decode only the generated continuation.
    response_ids = outputs[:, prompt_length:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)

Let's try generating an output using the model and one example from validation:

In [ ]:
# Compare the original request, expected JSON, and base-model output for one example.
example = calendar["validation"][0]
print("INPUT:")
print(example["utt"])

print("\nEXPECTED:")
print(target_json(example))

print("\nPREDICTED:")
print(json.loads(generate_extraction(model, tokenizer, extraction_messages(example["utt"]))[0]))

Now, we can combine the helper functions we've developed so far to go from utterances in a dataset to predictions from a model. Given a model, a tokenizer, and a dataset/split, it will loop through the latter and produce predictions.

In [ ]:
from tqdm.auto import tqdm

# Generate responses for a dataset in batches rather than one request at a time.
def predict_split(model, tokenizer, dataset, batch_size=32, max_new_tokens=256):
    # Preserve prediction order so outputs stay aligned with dataset rows.
    predictions = []
    # Left-pad inputs for batched generation with a causal language model.
    tokenizer.padding_side = "left"
    model.eval()

    # Build prompts for one batch, generate replies, and accumulate stripped outputs.
    for start in tqdm(range(0, len(dataset), batch_size)):
        batch = dataset[start:start + batch_size]
        # write your code here
        # Tip: use the helper functions developed above
        messages = ...
        responses = ...
        predictions.extend(response.strip() for response in responses)

    # Return one prediction per dataset example in original order.
    return predictions

Let's call it using our model and the validation set to get predictions for our baseline. The calendar validation subset is small enough to generate sequentially with this 360M model. This cell is the first moderately time-consuming evaluation step.

In [ ]:
baseline_validation = predict_split(model, tokenizer, calendar["validation"])

### 18.25.5 Structured-Output Evaluation

In a complex task such as this, we might have to measure several levels of success separately:
- valid JSON;
- valid schema;
- correct action;
- slot precision, recall, and F1;
- exact record match.

The helper functions below can be used to determine if a given output string is a valid JSON object and if it follows the predefined schema, or to flatten slots as tuples or canonicalize a record (to ensure slots are sorted alphabetically) for easier comparison.

In [ ]:
# List the only action values accepted by the extraction schema.
VALID_ACTIONS = {"set", "query", "remove"}

# Parse a generated string as JSON, returning None when parsing fails.
def parse_json_prediction(text):
    # Separate syntactic JSON failures from downstream schema failures.
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return None

# Check the required keys, action, slot names, and slot value types.
def valid_schema(record):
    if not isinstance(record, dict):
        return False

    if set(record.keys()) != {"action", "slots"}:
        return False

    # Reject unsupported actions rather than accepting arbitrary strings.
    if record["action"] not in VALID_ACTIONS:
        return False

    # Validate the slots object and every list of extracted strings.
    slots = record["slots"]

    if not isinstance(slots, dict):
        return False

    # Require a known slot label and a list containing only nonempty strings.
    for slot, values in slots.items():
        if slot not in allowed_slots:
            return False

        if not isinstance(values, list):
            return False

        if any(not isinstance(value, str) or not value.strip() for value in values):
            return False

    return True

# Flatten a prediction into unique (slot, value) pairs for extraction metrics.
def slot_pairs(record):
    if not isinstance(record, dict):
        return set()

    # Treat malformed or missing slot collections as empty.
    slots = record.get("slots", {})

    if not isinstance(slots, dict):
        return set()

    # Collect valid slot/value combinations, ignoring malformed values.
    pairs = set()
    for slot, values in slots.items():
        if not isinstance(values, list):
            continue

        for value in values:
            if isinstance(value, str) and value.strip():
                pairs.add((slot, value.strip()))

    return pairs


# Normalize slot ordering for exact-record comparisons independent of list order.
def canonicalize(record):
    # Sort slot keys and slot values without changing the action.
    return {
        "action": record["action"],
        "slots": {slot: sorted(values) for slot, values in sorted(record["slots"].items())},
    }

Let's try these functions on a synthetic example first.

In [ ]:
# Self-checks for parsing and schema validation.
# Test both JSON parsing and rejection of records with incorrect schema keys.
valid_example = {"action": "set", "slots": {"date": ["tomorrow"]}}
assert parse_json_prediction(json.dumps(valid_example)) == valid_example
# The synthetic field may not be in the calendar schema,
# so use the first discovered slot for a strict schema check.
schema_example = {"action": "set", "slots": {allowed_slots[0]: ["example"]}}
assert valid_schema(schema_example)
assert not valid_schema({"operation": "set", "slots": {}})
print("Structured-output checks passed.")

The helper functions are then used to evaluate a full set of predictions against their ground truth, producing a dictionary with accuracy results for JSON and schema validity, as well as a correct action being called; and precision, recall, and F1-score values for the predicted slots.

In [ ]:
import evaluate

# Load reusable metrics for binary success rates and slot extraction.
accuracy_metric = evaluate.load("accuracy")
slot_metrics = evaluate.combine(["precision", "recall", "f1"])

# Compute the fraction of 1-valued success indicators.
def binary_accuracy(values):
    # Compare the observed binary flags against an all-success reference.
    return accuracy_metric.compute(predictions=values, references=[1] * len(values))["accuracy"]

# Score JSON validity, schema validity, actions, slots, and complete records.
def evaluate_predictions(examples, predictions):
    # Store one success/failure flag per example for each record-level metric.
    json_valid = []
    schema_valid = []
    action_correct = []
    exact_records = []

    # Collect binary membership labels over the union of gold and predicted slot pairs.
    slot_references = []
    slot_predictions = []

    # Build each gold record and parse the corresponding generated response.
    for example, prediction in zip(examples, predictions):
        gold = build_target(example)
        parsed = parse_json_prediction(prediction)

        # Distinguish valid JSON from JSON that also satisfies the schema.
        is_json_valid = parsed is not None
        is_schema_valid = valid_schema(parsed) if is_json_valid else False

        # Record format, action, and exact-match successes separately.
        json_valid.append(int(is_json_valid))
        schema_valid.append(int(is_schema_valid))
        action_correct.append(int(is_json_valid and parsed.get("action") == gold["action"]))
        exact_records.append(int(is_schema_valid and canonicalize(parsed) == canonicalize(gold)))

        # Compare extracted slot/value pairs regardless of their positions in the JSON.
        gold_pairs = slot_pairs(gold)
        predicted_pairs = slot_pairs(parsed) if is_json_valid else set()

        # Turn each candidate pair into a binary prediction/reference for precision and recall.
        for pair in gold_pairs | predicted_pairs:
            slot_references.append(int(pair in gold_pairs))
            slot_predictions.append(int(pair in predicted_pairs))

    # Aggregate slot-level precision, recall, and F1 across examples.
    slot_scores = slot_metrics.compute(predictions=slot_predictions, references=slot_references)

    # Return all metric values under stable labels for before/after comparisons.
    return {
        "json_validity": binary_accuracy(json_valid),
        "schema_validity": binary_accuracy(schema_valid),
        "action_accuracy": binary_accuracy(action_correct),
        "slot_precision": slot_scores["precision"],
        "slot_recall": slot_scores["recall"],
        "slot_f1": slot_scores["f1"],
        "exact_record_accuracy": binary_accuracy(exact_records),
    }

Let's try these functions on a synthetic example as well:

In [ ]:
# Synthetic evaluator check.
# Create a perfectly predicted synthetic example to verify all evaluator metrics.
synthetic_examples = [{
    "intent": intent_feature.str2int("calendar_set"),
    "annot_utt": f"remind me [{allowed_slots[0]} : example]",
}]
synthetic_predictions = [
    json.dumps({"action": "set", "slots": {allowed_slots[0]: ["example"],}})
]
synthetic_metrics = evaluate_predictions(synthetic_examples, synthetic_predictions)
assert all(value == 1.0 for value in synthetic_metrics.values())
print("Evaluator check passed.")

Once the checks pass, we can call the evaluation function on our baseline predictions.

In [ ]:
# Evaluate the untuned base model on the validation split before training.
baseline_validation_metrics = evaluate_predictions(calendar["validation"], baseline_validation)
pd.Series(baseline_validation_metrics)

The results are certainly not very encouraging but, then again, this is the baseline. We can inspect the results in a more structured way using a dataframe:

In [ ]:
# Build an example-level table to inspect how the baseline model fails.
baseline_rows = []

# Compare each prediction with the corresponding gold record and schema.
for example, prediction in zip(calendar["validation"], baseline_validation):
    parsed = parse_json_prediction(prediction)
    gold = build_target(example)

    baseline_rows.append({
        "utterance": example["utt"],
        "gold": target_json(example),
        "prediction": prediction,
        "json_valid": parsed is not None,
        "schema_valid": valid_schema(parsed) if parsed is not None else False,
        "action_correct": isinstance(parsed, dict) and parsed.get("action") == gold["action"],
        "exact": valid_schema(parsed) and canonicalize(parsed) == canonicalize(gold) if parsed is not None else False,
    })

# Display a sample of the detailed baseline error labels.
baseline_errors = pd.DataFrame(baseline_rows)
baseline_errors.head(10)

The baseline table lets you distinguish formatting failures from semantic extraction failures. This matters because a model can identify the correct operation while still producing unusable structured output, especially if the generation is configured to produce outputs that are too short to include a fully-formed JSON object.

We also compute and keep the base-model test predictions now, but we do **not** inspect their metrics while making training decisions.

In [ ]:
# Cache base-model test predictions for comparison after training, without scoring them yet.
baseline_test = predict_split(model, tokenizer, calendar["test"])
print("Computed", len(baseline_test), "baseline test predictions.")

### 18.25.6 Preparing and Configuring (Q)LoRA

Now that the baseline has been established, we start working on fine-tuning the model. Since it was loaded as a 4-bit quantized model, we must **prepare** the model for quantized (k-bit) training. The PEFT package has a function especially developed for this purpose.

In [ ]:
from peft import prepare_model_for_kbit_training

# Prepare the quantized base model for adapter training while keeping base weights frozen.
# write your code here
model = ...
# Disable generation caching during gradient-based training.
model.config.use_cache = False

Next, we create a LoRA configuration object specifying rank, alpha, dropout, and target modules. We can use QLoRA-style `all-linear` targeting so the adapter is not tied to manually named projection layers. The configuration must be applied to the model, and we can double-check by inspecting the number of trainable parameters.

In [ ]:
from peft import LoraConfig, get_peft_model

# Define low-rank adapters for all linear layers in this causal language model.
# write your code here
lora_config = ...

# Attach the adapters and inspect how few parameters remain trainable.
# write your code here
model = ...
model.print_trainable_parameters()

The frozen base parameters still participate in every forward pass. They provide the pretrained language model. Only the low-rank adapter parameters receive optimizer updates.

### 18.26.6 Inspecting Sequence Length

The maximum sequence length is one of the most consequential choices when fine-tuning a model, as the memory requirements grow linearly with sequence length in the best case scenario, and quadratically in the worst. Therefore, the sequence length used should be as long as needed to cover the majority of cases, but not longer than necessary to avoid incurring excessive memory costs.

So, we inspect the distribution of the sequence lengths of our dataset in order to determine the sequence length that fits the absolute majority of the cases.

In [ ]:
# Count tokens in the full prompt-plus-completion chat sequence.
def formatted_length(example):
    # Include both the request and target JSON in the measured sequence.
    conversation = example["prompt"] + example["completion"]
    tokenized = tokenizer.apply_chat_template(conversation, tokenize=True, add_generation_prompt=False)
    return len(tokenized['input_ids'])

# Measure training example lengths to choose a safe sequence limit.
lengths = [formatted_length(example) for example in train_data]

# Summarize the length distribution, including upper percentiles.
length_summary = pd.Series(lengths).describe(percentiles=[0.90, 0.95, 0.99])
length_summary

In [ ]:
# MASSIVE calendar requests are short. We choose the smallest multiple of 32 that covers the observed maximum, capped at 256 tokens.
import math
# Calculate the sequence cap and count examples that would exceed it.
observed_max = max(lengths)
max_length = min(256, max(64, int(math.ceil(observed_max / 32) * 32)))
n_truncated = sum(length > max_length for length in lengths)
print("Observed maximum:", observed_max)
print("Chosen max_length:", max_length)
print("Training examples longer than max_length:", n_truncated)

If the final count above is non-zero, inspect those examples before training. Truncating the target JSON would corrupt the supervision signal.

### 18.26.7 Fine-Tuning the Model with `SFTTrainer`

Before training, we must configure the training arguments by creating an instance of `SFTConfig`.

In [ ]:
from trl import SFTConfig, SFTTrainer

# Inspect GPU capability to select the supported mixed-precision training mode.
major, minor = torch.cuda.get_device_capability()

# Configure LoRA training, effective batch size, precision, and evaluation cadence.
training_args = SFTConfig(
    output_dir="smollm2-calendar-qlora",
    # Set the number of passes over the training set.
    num_train_epochs=3,
    # Defines batch sizes
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    # Accumulate gradients to increase the effective batch size without extra memory.
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    # Learning rate
    learning_rate=2e-4,
    warmup_steps=3,
    # Use the maximum token length selected from observed examples.
    max_length=max_length,
    # Train on the assistant JSON completion rather than the input prompt.
    completion_only_loss=True,
    # Evaluate at the end of each epoch; do not save intermediate checkpoints here.
    eval_strategy="epoch",
    save_strategy="no",
    # Logging
    logging_steps=10,
    # Enable the compatible mixed-precision option for the available GPU.
    fp16=(compute_dtype == torch.float16) if major >= 8 else False,
    bf16=(compute_dtype == torch.bfloat16),
    # Reporting
    report_to="none",
    seed=42,
)

The trainer takes the model, the tokenizer, the datasets, and the training arguments.

In [ ]:
# Connect the model, datasets, tokenizer, and training configuration.
# write your code here
trainer = ...

We're ready to start the actual training. All it takes is a call to the trainer's `train()` method. It takes roughly 15 minutes in Colab.

In [ ]:
# Train the adapters and inspect the reported training metrics.
train_result = trainer.train()
train_result.metrics

Once training is finished, we can get the evaluation metrics by calling `evaluate()`.

In [ ]:
# Compute the final validation loss using the training objective.
eval_metrics = trainer.evaluate()
eval_metrics

A lower loss confirms that the adapter fits the token-level training objective, but it does **not** tell us whether the model now emits valid JSON or extracts the correct fields. Those are measured directly next.

At this point, we can also save the model and tokenizer to disk, so it can be reloaded and reused later on.

In [ ]:
import os

# Save the fine-tuned adapter and tokenizer without duplicating the base model.
adapter_dir = "smollm2-calendar-qlora-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# List saved artifacts and their sizes to inspect the adapter footprint.
for filename in sorted(os.listdir(adapter_dir)):
    path = os.path.join(adapter_dir, filename)
    if os.path.isfile(path):
        print(filename, f"{os.path.getsize(path) / 1024**2:.3f} MB")

Only the adapter and PEFT configuration need to represent the learned task-specific update. The full base checkpoint does not need to be duplicated for every specialization.

### 18.26.8 Evaluation After Fine-Tuning

We have already developed a helper function to evaluate the predictions of the baseline model. We can use it to evaluate the fine-tuned model now.

In [ ]:
# Reenable cache and switch to evaluation behavior before generating predictions.
model.config.use_cache = True
model.eval()

# Run the fine-tuned model on the same validation examples as the baseline.
fine_tuned_validation = predict_split(model, tokenizer, calendar["validation"])
fine_tuned_validation_metrics = evaluate_predictions(calendar["validation"], fine_tuned_validation)

# Place baseline and fine-tuned metrics side by side.
validation_comparison = pd.DataFrame({
    "Base": baseline_validation_metrics,
    "QLoRA": fine_tuned_validation_metrics,
})
validation_comparison

The most informative interpretation separates several effects:

- **JSON validity**: did the model learn the surface contract?
- **schema validity**: did it learn the exact interface?
- **action accuracy**: did it recognize the calendar operation?
- **slot F1**: did it extract the correct values?
- **exact record accuracy**: did the complete actionable record match?

High action accuracy alone is not enough: a conventional classifier could solve that narrower problem.

#### 18.26.8.1 Before/After Error Analysis

We can compare predictions for a few examples, before and after fine-tuning.

In [ ]:
# Build a record-by-record comparison of base and fine-tuned outputs.
comparison_rows = []

# Align each input with both predictions and its gold JSON.
for (example, base_prediction, tuned_prediction) in zip(calendar["validation"], baseline_validation, fine_tuned_validation):
    gold = build_target(example)
    base_parsed = parse_json_prediction(base_prediction)
    tuned_parsed = parse_json_prediction(tuned_prediction)

    # Check that a prediction satisfies the schema and matches all target fields.
    def is_exact(parsed):
        # Canonicalize records to ignore irrelevant ordering differences.
        return (parsed is not None and valid_schema(parsed) and canonicalize(parsed) == canonicalize(gold))

    # Record validity and exact-match flags for both models.
    comparison_rows.append({
        "utterance": example["utt"],
        "gold": target_json(example),
        "base_prediction": base_prediction,
        "tuned_prediction": tuned_prediction,
        "base_valid": base_parsed is not None and valid_schema(base_parsed),
        "tuned_valid": tuned_parsed is not None and valid_schema(tuned_parsed),
        "base_exact": is_exact(base_parsed),
        "tuned_exact": is_exact(tuned_parsed),
    })

# Create a table for targeted error inspection.
comparison_df = pd.DataFrame(comparison_rows)
comparison_df.head()

We can also split the changes among three classes: fixed by fine-tuning, still wrong, and regressions.

In [ ]:
# Show cases the adapter corrected relative to the baseline.
print("Fixed by fine-tuning:")
display(comparison_df[~comparison_df["base_exact"] & comparison_df["tuned_exact"]].head(5))
# Inspect requests that remain incorrect after fine-tuning.
print("\nStill wrong:")
display(comparison_df[~comparison_df["base_exact"] & ~comparison_df["tuned_exact"]].head(5))
# Identify previously correct predictions that became incorrect.
print("\nRegressions:")
display(comparison_df[comparison_df["base_exact"] & ~comparison_df["tuned_exact"]].head(5))

#### 18.26.8.2 Performance by Action

There are thee types of action in the calendar scenario: `query`, `set`, and `remove`. How did the model perform across these types?

In [ ]:
# Keep only examples whose gold intent corresponds to a particular action.
def subset_by_action(dataset, predictions, action):
    # Maintain aligned example and prediction subsets.
    selected_examples = []
    selected_predictions = []

    # Filter pairs together so evaluation order is preserved.
    for example, prediction in zip(dataset, predictions):
        if action_from_intent(example["intent"]) == action:
            selected_examples.append(example)
            selected_predictions.append(prediction)

    return (selected_examples, selected_predictions,)

# Score validation performance separately for each supported action.
action_rows = []

# Collect support, exact-record accuracy, and slot F1 per action.
for action in sorted(VALID_ACTIONS):
    examples, predictions = subset_by_action(calendar["validation"], fine_tuned_validation, action)
    metrics = evaluate_predictions(examples, predictions)
    action_rows.append({
        "action": action,
        "examples": len(examples),
        "exact_accuracy": metrics["exact_record_accuracy"],
        "slot_f1": metrics["slot_f1"],
    })

pd.DataFrame(action_rows)

#### 18.26.8.3 Performance by Slot Type

What about the slots? There are 20 possible allowed slots, and their number varies across examples, so this is a much harder task. Moreover, they are unevenly distributed. So, let's check the fine-tuned performance across slots.

In [ ]:
# Set up precision, recall, and F1 for each individual slot type.
slot_metrics = evaluate.combine(["precision", "recall", "f1"])

# Compute slot-value extraction performance separately for every allowed slot.
def per_slot_metrics(dataset, predictions):
    # Build a summary with one row per slot type.
    rows = []

    # Process a single slot label across the whole dataset.
    for slot in allowed_slots:
        references = []
        predicted = []

        # Collect the true and predicted values for this slot on each request.
        for example, prediction in zip(dataset, predictions):
            gold = build_target(example)
            parsed = parse_json_prediction(prediction)

            # Filter (slot, value) pairs down to values of the current slot.
            gold_values = {value for pair_slot, value in slot_pairs(gold) if pair_slot == slot}
            pred_values = set() if parsed is None else {
                value for pair_slot, value in slot_pairs(parsed) if pair_slot == slot
            }

            # Count both missed true values and extra predicted values.
            for value in gold_values | pred_values:
                references.append(int(value in gold_values))
                predicted.append(int(value in pred_values))

        # Compute metrics when the slot has candidates; otherwise use zero scores.
        if references:
            scores = slot_metrics.compute(predictions=predicted, references=references)
        else:
            scores = {"precision": 0.0, "recall": 0.0, "f1": 0.0}

        # Store metrics with the number of gold values as support.
        rows.append({"slot": slot, **scores, "support": sum(references)})

    # Return slot types ordered by F1 so weak performers are easy to spot.
    return pd.DataFrame(rows).sort_values("f1", ascending=True)

# View only slot types represented in the validation references.
slot_results = per_slot_metrics(calendar["validation"], fine_tuned_validation)
slot_results.query('support > 0')

Low-support slots should be interpreted carefully: an F1 score based on only a handful of examples is much less stable than one based on many occurrences.

#### 18.26.8.4 Final Test Evaluation

Only now do we calculate final test metrics for the fine-tuned model and compare it to the baseline results we have computed (and have not inspected) earlier.

In [ ]:
# Generate predictions on held-out test requests only after model selection is finished.
fine_tuned_test = predict_split(reloaded_model, tokenizer, calendar["test"])
# Evaluate both saved baseline outputs and reloaded adapter outputs.
base_test_metrics = evaluate_predictions(calendar["test"], baseline_test)
fine_tuned_test_metrics = evaluate_predictions(calendar["test"], fine_tuned_test)
# Compare the two sets of final test metrics in one table.
test_comparison = pd.DataFrame({
    "Base": base_test_metrics,
    "QLoRA": fine_tuned_test_metrics,
})
test_comparison

#### 18.26.8.5 Hallucination Analysis

In extraction, invented fields are particularly important because downstream software may act on them. A predicted `(slot, value)` pair that is absent from the gold record is a false positive and therefore lowers slot precision. Let's see what kind of slots, if any, were hallucinated by the model.

In [ ]:
# Find predicted slot/value pairs that were absent from the gold extraction.
hallucination_rows = []

# Compare extracted pairs for every test request.
for example, prediction in zip(calendar["test"], fine_tuned_test):
    gold = build_target(example)
    parsed = parse_json_prediction(prediction)
    predicted_pairs = slot_pairs(parsed) if parsed is not None else set()
    gold_pairs = slot_pairs(gold)
    # Treat extra predicted pairs as candidates for hallucination analysis.
    extra_pairs = predicted_pairs - gold_pairs
    if extra_pairs:
        hallucination_rows.append({
            "utterance": example["utt"],
            "gold": target_json(example),
            "prediction": prediction,
            "extra_pairs": sorted(extra_pairs),
        })

# Show requests with extra extracted values for manual inspection.
hallucinations = pd.DataFrame(hallucination_rows)
hallucinations.head(10)

Not every apparent false positive is necessarily a nonsensical hallucination. Inspect examples and distinguish:

- a genuinely invented value;
- a value present in the utterance but not annotated by MASSIVE;
- a correct span assigned to the wrong slot type;
- a potentially questionable benchmark annotation.

#### 18.26.8.6 Testing Original Requests

Finally, let's come up with some original requests of our own and try and find out if the model successfully handles them.

In [ ]:
# Define additional calendar requests, including one with missing details.
new_requests = [
    "Schedule lunch with Maria next Friday at noon.",
    "What's on my calendar tomorrow afternoon?",
    "Cancel my dentist appointment on Tuesday.",
    "Schedule a meeting.",
    "Remove my appointment with Alex tonight.",
]

# Run the trained extractor on each custom request and display its output.
for utterance in new_requests:
    print("INPUT:")
    print(utterance)

    print("\nOUTPUT:")
    print(generate_extraction(reloaded_model, tokenizer, extraction_messages(utterance)))
    print("\n" + "=" * 60)

The incomplete request `Schedule a meeting.` is particularly revealing. The model should not invent a date, time, person, or location simply because those fields are common in its training examples.

#### 18.26.8.7 Challenge: Out-of-Scope Requests

Our schema contains only `set`, `query` and `remove` as valid actions. So, there is **no correct action** for a request such as "What's the weather tomorrow?"

A production design should normally add an `out_of_scope` action and include negative examples from non-calendar MASSIVE scenarios during training. That change should also add an out-of-scope detection metric. This is a schema/data problem rather than something that can be fixed reliably with a generation parameter.